# Generating Simple Embendding

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
from openai import OpenAI
client = OpenAI()

In [8]:
text=''' This is a random text
OpenAI o3 is a GPT model developed by OpenAI
'''


text = text.replace('\n', ' ')

emd = client.embeddings.create(
    input=text,
    model='text-embedding-3-small'
)

print(emd.data[0].embedding)

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.02166748046875, 0.020904541015625, 0.046875, -0.0236358642578125, 0.003246307373046875, -0.054443359375, -0.01425933837890625, 0.0181732177734375, -0.0007915496826171875, -0.017486572265625, 0.03485107421875, -0.004306793212890625, -0.01837158203125, 0.0094757080078125, -0.0239410400390625, -0.0139923095703125, -0.0195770263671875, -0.09002685546875, -0.039306640625, 0.0013523101806640625, 0.0256500244140625, 0.0229034423828125, 0.00798797607421875, -0.015167236328125, 0.0116729736328125, 0.003917694091796875, 0.0206451416015625, 0.0239410400390625, 0.006877899169921875, -0.0333251953125, 0.033416748046875, -0.0238800048828125, -0.07525634765625, -1.1980533599853516e-05, -0.021148681640625, -0.00046181678771972656, -0.005756378173828125, 0.018951416015625, -0.0213775634765625, 0.036285400390625, 0.0012683868408203125, -0.01551055908203125, -0.00551605224609375, 0.033721923828125, -0.0171051025390625, 0.0109100341796875, 0.0205383300

In [9]:
pip install pandas numpy -q

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
df=pd.read_csv('words.csv')
df=df.sample(frac=1)
df

,word
1,banana
9,data
0,apple
8,text
7,model
6,vector
2,computer
3,python
4,openai
5,embedding


def get_embendding(text, model='text-embedding-3-small'):
    client = OpenAI()
    text = text.replace('\n', ' ')

    emd = client.embeddings.create(
        input=text,
        model='text-embedding-3-small'
    )

    return emd.data[0].embedding

In [14]:
df['embedding']=df['word'].apply(lambda x: get_embendding(x))
df

,word,embedding
1,banana,"[0.00855255126953125, -0.0240631103515625, -0...."
9,data,"[0.0243682861328125, 0.012298583984375, 0.0289..."
0,apple,"[0.0176544189453125, -0.0168609619140625, -0.0..."
8,text,"[-0.005947113037109375, 0.029541015625, -0.038..."
7,model,"[-0.00667572021484375, 0.0031375885009765625, ..."
6,vector,"[-0.007465362548828125, 0.0069427490234375, -0..."
2,computer,"[-0.0109100341796875, 0.0078277587890625, -0.0..."
3,python,"[-0.01039886474609375, -0.012481689453125, 0.0..."
4,openai,"[0.00598907470703125, -0.00870513916015625, 0...."
5,embedding,"[-0.0070037841796875, 0.0122833251953125, 0.03..."


In [15]:
df.to_csv('words-embenddings.csv',index=False)

## Estimating Embendding Cost with tiktokem

In [16]:
pip install tiktoken -q

Note: you may need to restart the kernel to use updated packages.


In [18]:
import tiktoken
import pandas as pd
df=pd.read_csv('words-embenddings.csv')

In [19]:
df

,word,embedding
0,banana,"[0.00855255126953125, -0.0240631103515625, -0...."
1,data,"[0.0243682861328125, 0.012298583984375, 0.0289..."
2,apple,"[0.0176544189453125, -0.0168609619140625, -0.0..."
3,text,"[-0.005947113037109375, 0.029541015625, -0.038..."
4,model,"[-0.00667572021484375, 0.0031375885009765625, ..."
5,vector,"[-0.007465362548828125, 0.0069427490234375, -0..."
6,computer,"[-0.0109100341796875, 0.0078277587890625, -0.0..."
7,python,"[-0.01039886474609375, -0.012481689453125, 0.0..."
8,openai,"[0.00598907470703125, -0.00870513916015625, 0...."
9,embedding,"[-0.0070037841796875, 0.0122833251953125, 0.03..."


In [22]:
words=list(df['word'])
enc=tiktoken.encoding_for_model('text-embedding-3-small')
total_token=sum([len(enc.encode(word)) for word in words])
print(f'Total Tokens: {total_token}')

Total Tokens: 11


In [24]:
cost_per_token=0.02/1_000_000
est_cost=total_token * cost_per_token
print(f'est total cost {est_cost: .10f}')

est total cost  0.0000002200


## Performing Semantic Searches

In [28]:
pip install -q numpy

Note: you may need to restart the kernel to use updated packages.


In [29]:
import  pandas as pd
import numpy as np
df=pd.read_csv('words-embenddings.csv')

In [30]:
df['embedding']=df['embedding'].apply(eval).apply(np.array)


In [32]:
search_term='red'
search_term_vector= get_embendding(search_term)

In [33]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

In [36]:
df['similar']=  df['embedding'].apply(lambda x: cosine_similarity(x, search_term_vector))
df.sort_values('similar', ascending=False)

,word,embedding,similar
2,apple,"[0.0176544189453125, -0.0168609619140625, -0.0...",0.324858
4,model,"[-0.00667572021484375, 0.0031375885009765625, ...",0.322956
5,vector,"[-0.007465362548828125, 0.0069427490234375, -0...",0.305788
1,data,"[0.0243682861328125, 0.012298583984375, 0.0289...",0.302794
3,text,"[-0.005947113037109375, 0.029541015625, -0.038...",0.268638
0,banana,"[0.00855255126953125, -0.0240631103515625, -0....",0.242090
6,computer,"[-0.0109100341796875, 0.0078277587890625, -0.0...",0.229030
8,openai,"[0.00598907470703125, -0.00870513916015625, 0....",0.228124
9,embedding,"[-0.0070037841796875, 0.0122833251953125, 0.03...",0.180889
7,python,"[-0.01039886474609375, -0.012481689453125, 0.0...",0.170303


In [37]:
v1= df['embedding'].iloc[4]
v2= df['embedding'].iloc[8]
v=v1+v2
v

array([-0.00068665, -0.00556755,  0.01335144, ..., -0.00547791,
       -0.01185608, -0.05653381], shape=(1536,))